# **STEP 1: TRANSFORMASI MATRIKS DASAR**

In [24]:
import math
class TransformationMatrices:
    def __init__(self, data):
        self.data = data
        if isinstance(data[0], list):
            self.shape = (len(data), len(data[0]))
        else:
            self.shape = (1, len(data))

    def __repr__(self):
        rows_str = "\n  ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n  {rows_str}"

    def rotation_2d(self, theta):
        c, s = math.cos(theta), math.sin(theta)
        return [[c, -s], [s, c]]

    def scaling_2d(self, sx, sy):
        return [[sx, 0], [0, sy]]

    def shearing_2d(self, kx, ky):
        return [[1, kx], [ky, 1]]

    def reflection_x(self):
        return [[1, 0], [0, -1]]

    def reflection_y(self):
        return [[-1, 0], [0, 1]]

    def mat_vec_mul(self, matrix, vector):
        return [
            sum(matrix[i][j] * vector[j] for j in range(len(vector)))
            for i in range(len(matrix))
        ]

    def mat_mul(self, a, b):
        return [
            [
                sum(a[i][k] * b[k][j] for k in range(len(b)))
                for j in range(len(b[0]))
            ]
            for i in range(len(a))
        ]


In [ ]:
point = [1,2]
angle = math.pi / 4

tf_mat = TransformationMatrices(point)

rotated = tf_mat.mat_vec_mul(tf_mat.rotation_2d(angle), point)
print(f"Rotate (1,0) by 45 deg: ({rotated[0]:.4f}, {rotated[1]:.4f})")

scaled = tf_mat.mat_vec_mul(tf_mat.scaling_2d(2, 3), [1.0, 1.0])
print(f"Scale (1,1) by (2,3): ({scaled[0]:.1f}, {scaled[1]:.1f})")

sheared = tf_mat.mat_vec_mul(tf_mat.shearing_2d(1, 0), [1.0, 1.0])
print(f"Shear (1,1) kx=1: ({sheared[0]:.1f}, {sheared[1]:.1f})")

reflected = tf_mat.mat_vec_mul(tf_mat.reflection_y(), [2.0, 1.0])
print(f"Reflect (2,1) across y: ({reflected[0]:.1f}, {reflected[1]:.1f})")

Rotate (1,0) by 45 deg: (-0.7071, 2.1213)
Scale (1,1) by (2,3): (2.0, 3.0)
Shear (1,1) kx=1: (2.0, 1.0)
Reflect (2,1) across y: (-2.0, 1.0)


0.7853981633974483

# **STEP 2 : KOMPOSISI TRANSFORMASI**

In [30]:
R = tf_mat.rotation_2d(math.pi / 2)
S = tf_mat.scaling_2d(2, 0.5)

rotate_then_scale = tf_mat.mat_mul(S, R)
scale_then_rotate = tf_mat.mat_mul(R, S)

point = [1.0, 0.0]
result1 = tf_mat.mat_vec_mul(rotate_then_scale, point)
result2 = tf_mat.mat_vec_mul(scale_then_rotate, point)

print(f"Rotate 90 then scale: ({result1[0]:.2f}, {result1[1]:.2f})")
print(f"Scale then rotate 90: ({result2[0]:.2f}, {result2[1]:.2f})")
print(f"Same? {result1 == result2}")

Rotate 90 then scale: (0.00, 0.50)
Scale then rotate 90: (0.00, 2.00)
Same? False


In [ ]:
class EigenVector(TransformationMatrices):
    def __init__(self, matrix):
        self.matrix = matrix
        self.eigenvalues = []

    def eigenvalues_2x2(self):
        a, b = self.matrix[0]
        c, d = self.matrix[1]
        trace = a + d
        det = a * d - b * c
        discriminant = trace ** 2 - 4 * det

        if discriminant < 0:
            real = trace / 2
            imag = (-discriminant) ** 0.5 / 2
            return (complex(real, imag), complex(real, -imag))

        sqrt_disc = discriminant ** 0.5
        self.eigenvalues = ((trace + sqrt_disc) / 2, (trace - sqrt_disc) / 2)
        return self.eigenvalues

    def eigenvector_2x2(self, eigenvalue):
        a, b = self.matrix[0]
        c, d = self.matrix[1]

        if abs(b) > 1e-10:
            v = [b, eigenvalue - a]
        elif abs(c) > 1e-10:
            v = [eigenvalue - d, c]
        else:
            if abs(a - eigenvalue) < 1e-10:
                v = [1, 0]
            else:
                v = [0, 1]

        mag = (v[0] ** 2 + v[1] ** 2) ** 0.5
        vals = [v[0] / mag, v[1] / mag]
        return vals


A = EigenVector([[2, 1], [1, 2]])
vals = A.eigenvalues_2x2()

for val in vals:
    vec = A.eigenvector_2x2(val)
    result = A.mat_vec_mul(A.matrix, vec)
    scaled = [val * vec[0], val * vec[1]]

    print(f"  lambda={val:.1f}, v={[round(x,4) for x in vec]}")
    print(f"    A@v = {[round(x,4) for x in result]}")
    print(f"    l*v = {[round(x,4) for x in scaled]}")


  lambda=3.0, v=[0.7071, 0.7071]
    A@v = [2.1213, 2.1213]
    l*v = [2.1213, 2.1213]
  lambda=1.0, v=[0.7071, -0.7071]
    A@v = [0.7071, -0.7071]
    l*v = [0.7071, -0.7071]


# **STEP 4 : DETERMINAN SEBAGAI FAKTOR SKALA VOLUME**

In [38]:
def det_2x2(matrix):
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

print(f"det(rotation 45) = {det_2x2(tf_mat.rotation_2d(math.pi/4)):.4f}")
print(f"det(scale 2,3)   = {det_2x2(tf_mat.scaling_2d(2, 3)):.1f}")
print(f"det(shear kx=1)  = {det_2x2(tf_mat.shearing_2d(1, 0)):.1f}")
print(f"det(reflect y)   = {det_2x2(tf_mat.reflection_y()):.1f}")

singular = [[1, 2], [2, 4]]
print(f"det(singular)     = {det_2x2(singular):.1f}")
print("Singular: columns are proportional, space collapses to a line.")

det(rotation 45) = 1.0000
det(scale 2,3)   = 6.0
det(shear kx=1)  = 1.0
det(reflect y)   = -1.0
det(singular)     = 0.0
Singular: columns are proportional, space collapses to a line.


# **LATIHAN SOAL**

**01.** Terapkan rotasi, penskalaan, dan pergeseran pada persegi satuan (sudut di [0,0], [1,0], [1,1], [0,1]). 
Cetak sudut yang telah ditransformasi untuk masing-masing persegi. Verifikasi bahwa rotasi mempertahankan jarak antar sudut.

In [49]:
persegi = [
    [0, 0],
    [1, 0],
    [1, 1],
    [0, 1]
]

trans = TransformationMatrices(persegi)
rotation = []
scaler = []
shering = []

rotation_matrix = trans.rotation_2d(math.radians(45))
scaling_matrix = trans.scaling_2d(2, 3)
shearing_matrix = trans.shearing_2d(1, 0)

for point in persegi:
    rotation.append(trans.mat_vec_mul(rotation_matrix,point))
    scaler.append(trans.mat_vec_mul(scaling_matrix,point))
    shering.append(trans.mat_vec_mul(shearing_matrix,point))

def distance(p1, p2):
    return math.sqrt(
        (p2[0] - p1[0])**2 +
        (p2[1] - p1[1])**2
    )

jarak_awal = distance(persegi[0], persegi[1])
jarak_rotasi = distance(rotation[0], rotation[1])

print("Koordinat Awal:", persegi)
print("\nHasil Rotasi:", rotation)
print("\nHasil Scaling:", scaler)
print("\nHasil Shearing:", shering)

print("\n--- Verifikasi Jarak ---")
print("Jarak sebelum rotasi:", jarak_awal)
print("Jarak setelah rotasi:", jarak_rotasi)
print("Rotasi mempertahankan jarak:",math.isclose(jarak_awal, jarak_rotasi))

Koordinat Awal: [[0, 0], [1, 0], [1, 1], [0, 1]]

Hasil Rotasi: [[0.0, 0.0], [0.7071067811865476, 0.7071067811865476], [0.0, 1.4142135623730951], [-0.7071067811865476, 0.7071067811865476]]

Hasil Scaling: [[0, 0], [2, 0], [2, 3], [0, 3]]

Hasil Shearing: [[0, 0], [1, 0], [2, 1], [1, 1]]

--- Verifikasi Jarak ---
Jarak sebelum rotasi: 1.0
Jarak setelah rotasi: 1.0
Rotasi mempertahankan jarak: True


**02.** Carilah nilai eigen dari matriks [[4, 2], [1, 3]] secara manual menggunakan persamaan karakteristik. Kemudian verifikasi dengan fungsi yang Anda buat dari awal dan dengan NumPy.

In [60]:
import numpy as np

matrix_a = [[4,2],[1,3]]
eigen_matrik = EigenVector(matrix_a)

vals_manual = eigen_matrik.eigenvalues_2x2()
vals_numpy,_ = np.linalg.eig(matrix_a)

print("Eigenvalue Manual :", vals_manual)
print("Eigenvalue NumPy   :", vals_numpy)

print("\nVerifikasi:")
print(np.allclose(
    sorted(vals_manual),
    sorted(vals_numpy)
))

Eigenvalue Manual : (5.0, 2.0)
Eigenvalue NumPy   : [5. 2.]

Verifikasi:
True


**03.** Buat komposisi tiga transformasi (rotasi 30 derajat, skala dengan [1,5, 0,8], geser dengan kx=0,3) dan terapkan pada 8 titik yang tersusun dalam lingkaran. Cetak koordinat sebelum dan sesudah. ​​Hitung determinan matriks komposisi dan verifikasi bahwa hasilnya sama dengan hasil perkalian determinan masing-masing transformasi.

In [72]:
points = []
radius = 1
for i in range(8):
    theta = 2 * math.pi * i / 8
    x = radius * math.cos(theta)
    y = radius * math.sin(theta)
    points.append([x,y])

def determinant(matrix):
    a,b = matrix[0]
    c,d = matrix[1]
    return a * d - c * b

composision = TransformationMatrices(points)

rotation = composision.rotation_2d(math.radians(30))
scaling = composision.scaling_2d(1.5,0.8)
shering = composision.shearing_2d(0.3,0)

sr = composision.mat_mul(scaling,rotation)
srt = composision.mat_mul(shering,sr)

result = []
for point in points:
    result.append(composision.mat_vec_mul(srt,point))
det_composision = determinant(srt)

det_rotation = determinant(rotation)
det_scaling = determinant(scaling)
det_shering = determinant(shering)
det_product = (det_rotation * det_shering * det_scaling)

print("Determinan Komposisi:", det_composision)
print("Perkalian Determinan:", det_product)
print("Verifikasi:", math.isclose(det_composision, det_product))

Determinan Komposisi: 1.2
Perkalian Determinan: 1.2000000000000002
Verifikasi: True


In [73]:
print("\n=== Koordinat Sebelum Transformasi ===")
for i, point in enumerate(points):
    print(f"Titik {i+1}: {point}")

print("\n=== Koordinat Setelah Transformasi ===")
for i, point in enumerate(result):
    print(f"Titik {i+1}: {point}")


=== Koordinat Sebelum Transformasi ===
Titik 1: [1.0, 0.0]
Titik 2: [0.7071067811865476, 0.7071067811865476]
Titik 3: [6.123233995736766e-17, 1.0]
Titik 4: [-0.7071067811865475, 0.7071067811865476]
Titik 5: [-1.0, 1.2246467991473532e-16]
Titik 6: [-0.7071067811865477, -0.7071067811865475]
Titik 7: [-1.8369701987210297e-16, -1.0]
Titik 8: [0.7071067811865474, -0.7071067811865477]

=== Koordinat Setelah Transformasi ===
Titik 1: [1.419038105676658, 0.39999999999999997]
Titik 2: [0.6200507659631576, 0.7727406610312547]
Titik 3: [-0.5421539030917345, 0.692820323027551]
Titik 4: [-1.3867721686089973, 0.20705523608201676]
Titik 5: [-1.419038105676658, -0.39999999999999986]
Titik 6: [-0.6200507659631578, -0.7727406610312547]
Titik 7: [0.5421539030917344, -0.6928203230275511]
Titik 8: [1.386772168608997, -0.20705523608201692]
